# CloudTrend 한국·ETF Drive 자료수집기 v7 — 검증용 별도 버전

기존 v6.1·기존 실행기·Supabase 원본을 수정하지 않습니다. 기본은 **읽기 검증만** 수행합니다.

- 과거 원천: 고정 manifest + 독립 ZIP 5개, 등록 순서와 원본 SHA256 확인
- 수집: 기존 Toss 수정주가·KRX 공식 API 환경, 종목별 누락 구간 + 겹침 구간
- 보관: `한국시장/01_원천데이터/drive_history_v7/runs/<runId>/`
- 공식 필드 병합·점수 계산: 새 GitHub Actions에서 기존 엔진 사용
- **홈페이지 게시와 baseline/latest 승격은 아직 비활성입니다.** 실제 Actions 인증·운영 지수 동등성·웹 입력 identity·여러 작성자의 경쟁 제어 검증이 남았습니다.
- 미국은 별도 장기 원천 확인이 필요하여 이 노트북에서 실행하지 않습니다.

새 수집은 기준 baseline 이후의 거래일만 지원합니다. 같은 runId로 재개하면 완료 종목 CSV의 hash를 다시 확인합니다. 수정주가 변화가 감지되면 완료 처리하지 않고 `corrections.json`에 기록합니다.


In [ ]:
#@title 1. 실행 설정
DO_COLLECT = False #@param {type:"boolean"}
DATA_DATE = "" #@param {type:"string"}
# DATA_DATE: 실제 완료 거래일 YYYY-MM-DD. 공휴일/주말을 영업일로 추정하지 않습니다.
RUN_ID = "" #@param {type:"string"}
# 중단 재개 시 이전 출력의 runId를 입력하세요. 비우면 새 ID입니다.
OVERLAP_SESSIONS = 5 #@param {type:"integer"}
EXCLUDED_SYMBOLS = [] # 이미 제외한 종목을 명시하면 자동 재편입하지 않습니다.


In [ ]:
#@title 2. 설치 및 Google Drive 연결
%pip install -q pandas==2.2.3 requests==2.32.5 beautifulsoup4==4.13.5 tqdm==4.67.1 ipython==8.37.0
from google.colab import drive, userdata
from pathlib import Path
import os, sys, json, hashlib, uuid
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
BASELINE = Path('/content/drive/MyDrive/CloudTrend/한국시장/01_원천데이터/screening_history_baseline_20261107_v1')
RUN_ROOT = Path('/content/drive/MyDrive/CloudTrend/한국시장/01_원천데이터/drive_history_v7')
RUN_ID = RUN_ID.strip() or uuid.uuid4().hex
print('runId:', RUN_ID)
print('중간 저장:', RUN_ROOT / 'runs' / RUN_ID)


In [ ]:
#@title 3. 독립 v7 코드 준비 (이 노트북에 포함된 검토 대상 코드)
FILES = {'history.py': '"""Immutable KR/ETF Drive history transport. No Supabase source fallback; no executable archives."""\nfrom __future__ import annotations\nimport argparse, csv, hashlib, io, json, os, re, stat, tempfile, time, urllib.request, urllib.parse, urllib.error, zipfile\nfrom pathlib import Path, PurePosixPath\n\nBASELINE_ID = \'1lriFdL0lU9gu0-wlRx6Yc0NMPLAqh52D\'\nBASELINE_SHA = \'d0cb8477ca6f94a5ebbcbd8e79928635222e46f20e116fafb902c7763e0bfe92\'\nOWNER = \'bdfc8818-33a7-4030-9dbd-ecad39f223ac\'\nMAX_PACKAGE = 80 * 1024**2\nMAX_EXPANDED = 400 * 1024**2\n\ndef require(ok, message):\n    if not ok: raise ValueError(message)\n\ndef sha(data): return hashlib.sha256(data).hexdigest()\ndef atomic(path, data):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.NamedTemporaryFile(dir=path.parent, delete=False) as f:\n        f.write(data); f.flush(); os.fsync(f.fileno()); tmp = f.name\n    os.replace(tmp, path)\n\ndef checked(data, entry):\n    require(len(data) == entry[\'bytes\'] and sha(data) == entry[\'sha256\'].removeprefix(\'sha256:\'), \'SHA256/size mismatch\')\n    return data\n\ndef safe_member(name):\n    p = PurePosixPath(name)\n    require(not p.is_absolute() and \'..\' not in p.parts and \'\\\\\' not in name and str(p) == name, \'Unsafe archive path\')\n    return name\n\nclass Drive:\n    """Refresh on expiry/401; never persist bearer tokens or raw provider errors."""\n    def __init__(self, credentials=None):\n        self.credentials = credentials or json.loads(os.environ.get(\'GOOGLE_DRIVE_CREDENTIALS_JSON\', \'{}\'))\n        require(self.credentials.get(\'type\') == \'authorized_user\', \'GOOGLE_DRIVE_CREDENTIALS_JSON must contain refreshable authorized_user credentials\')\n        self.token, self.until, self.reads = \'\', 0, []\n    def refresh(self):\n        c = self.credentials\n        body = urllib.parse.urlencode({k:c[k] for k in (\'client_id\',\'client_secret\',\'refresh_token\')} | {\'grant_type\':\'refresh_token\'}).encode()\n        try:\n            with urllib.request.urlopen(urllib.request.Request(\'https://oauth2.googleapis.com/token\', data=body), timeout=60) as r: result = json.load(r)\n        except urllib.error.HTTPError as e: raise RuntimeError(f\'Drive OAuth refresh failed ({e.code}); renew authorized_user consent\') from None\n        self.token, self.until = result[\'access_token\'], time.monotonic() + int(result.get(\'expires_in\',3600)) - 120\n    def call(self, route, *, query=None, data=None, method=None, content_type=None, upload=False, limit=MAX_PACKAGE):\n        require(route == \'files\' or re.fullmatch(r\'files/[A-Za-z0-9_-]{10,200}\',route), \'Invalid Drive route\')\n        url = \'https://www.googleapis.com/\' + (\'upload/drive/v3/\' if upload else \'drive/v3/\') + route\n        if query: url += \'?\' + urllib.parse.urlencode(query)\n        for attempt in range(2):\n            if time.monotonic() >= self.until: self.refresh()\n            headers = {\'Authorization\':\'Bearer \'+self.token}\n            if content_type: headers[\'Content-Type\'] = content_type\n            try:\n                with urllib.request.urlopen(urllib.request.Request(url, data=data, method=method, headers=headers), timeout=180) as r:\n                    raw = r.read(limit+1); require(len(raw)<=limit,\'Drive response limit exceeded\'); return raw\n            except urllib.error.HTTPError as e:\n                if e.code == 401 and attempt == 0: self.until=0; continue\n                raise RuntimeError(f\'Drive request failed ({e.code})\') from None\n    def download(self, file_id):\n        raw = self.call(\'files/\'+file_id, query={\'alt\':\'media\',\'supportsAllDrives\':\'true\'})\n        self.reads.append({\'backend\':\'drive\',\'fileId\':file_id,\'bytes\':len(raw)})\n        return raw\n    def upload(self, folder, name, raw):\n        require(re.fullmatch(r\'[A-Za-z0-9_-]{10,200}\',folder),\'Invalid folder ID\')\n        boundary = \'cloudtrend_\'+sha(raw)\n        body = (f\'--{boundary}\\r\\nContent-Type: application/json\\r\\n\\r\\n\'.encode()+json.dumps({\'name\':name,\'parents\':[folder]}).encode()+\n                f\'\\r\\n--{boundary}\\r\\nContent-Type: application/octet-stream\\r\\n\\r\\n\'.encode()+raw+f\'\\r\\n--{boundary}--\\r\\n\'.encode())\n        result=json.loads(self.call(\'files\',query={\'uploadType\':\'multipart\',\'fields\':\'id,name,webViewLink\'},data=body,upload=True,content_type=f\'multipart/related; boundary={boundary}\'))\n        require(sha(self.download(result[\'id\']))==sha(raw),\'Drive readback mismatch\')\n        return result\n\ndef validate_manifest(raw, digest=BASELINE_SHA):\n    require(sha(raw)==digest,\'Baseline manifest hash mismatch\')\n    m=json.loads(raw)\n    require(m[\'schemaVersion\']==\'cloudtrend-drive-history-baseline-v1\' and m[\'market\']==\'KR_ETF\' and m[\'immutable\'] is True,\'Unsupported baseline\')\n    sources=m[\'sources\']; require(len(sources)==52,\'Expected pinned 52 sources\')\n    require([s[\'ordinal\'] for s in sources]==list(range(len(sources))),\'Source ordinal mismatch\')\n    records=[s[\'sourceRecord\'] for s in sources]\n    key=lambda r:(r[\'activated_at\'] is not None, r[\'activated_at\'] or \'\',r[\'created_at\'],r[\'id\'])\n    require(records==sorted(records,key=key),\'Registry precedence changed\')\n    require(len({r[\'id\'] for r in records})==len(records),\'Duplicate source IDs\')\n    packages={p[\'name\']:p for p in m[\'packages\']}; require(len(packages)==len(m[\'packages\']),\'Duplicate packages\')\n    for s in sources:\n        r=s[\'sourceRecord\']; safe_member(s[\'member\'])\n        require(s[\'member\']==\'objects/\'+s[\'sourcePath\'] and r[\'storage_path\']==s[\'sourcePath\'],\'Source path mismatch\')\n        require(r[\'user_id\']==OWNER and r[\'status\']==\'active\' and r[\'source_type\']==\'screening\',\'Unexpected source owner/type/status\')\n        require(r[\'file_hash\']==\'sha256:\'+s[\'sha256\'] and r[\'file_size_bytes\']==s[\'bytes\'],\'Registered byte evidence mismatch\')\n        require(s[\'packageName\'] in packages,\'Missing package descriptor\')\n    return m\n\ndef prepare_baseline(output, *, mounted=None, local_manifest=None, package_dir=None, drive=None):\n    """Recheck every cache hit. Extract one independent ZIP at a time, only expected members."""\n    out=Path(output); out.mkdir(parents=True,exist_ok=True)\n    if mounted: local_manifest=Path(mounted)/\'baseline-manifest.json\'; package_dir=Path(mounted)\n    raw=Path(local_manifest).read_bytes() if local_manifest else drive.download(BASELINE_ID)\n    m=validate_manifest(raw); total=0\n    for p in m[\'packages\']:\n        cache=out/\'packages\'/p[\'name\']\n        if package_dir: data=checked((Path(package_dir)/p[\'name\']).read_bytes(),p)\n        elif cache.exists(): data=checked(cache.read_bytes(),p)\n        else: data=checked(drive.download(p[\'fileId\']),p); atomic(cache,data)\n        require(len(data)<=MAX_PACKAGE,\'Package limit exceeded\')\n        expected={s[\'member\']:s for s in m[\'sources\'] if s[\'packageName\']==p[\'name\']}\n        with zipfile.ZipFile(io.BytesIO(data)) as z:\n            infos=z.infolist(); names=[i.filename for i in infos]\n            require(len(names)==len(set(names)) and set(names)==set(expected),\'Duplicate/unexpected/missing ZIP member\')\n            require(sum(i.file_size for i in infos)==p[\'uncompressedBytes\'] and len(infos)==p[\'memberCount\'],\'ZIP inventory mismatch\')\n            total+=sum(i.file_size for i in infos); require(total<=MAX_EXPANDED,\'ZIP expansion limit exceeded\')\n            for i in infos:\n                safe_member(i.filename); require(not stat.S_ISLNK(i.external_attr>>16),\'ZIP symlink forbidden\')\n                require(i.file_size==expected[i.filename][\'bytes\'] and i.file_size<=50*1024**2,\'Member size limit exceeded\')\n                data=checked(z.read(i),expected[i.filename]) # CRC checked by zipfile\n                atomic(out/i.filename,data)\n    atomic(out/\'baseline-manifest.json\',raw)\n    receipt={\'version\':\'drive-baseline-verification-v1\',\'manifestSha256\':sha(raw),\'sourceCount\':len(m[\'sources\']),\n             \'sourceBytes\':total,\'orderedSourceHashes\':[s[\'sha256\'] for s in m[\'sources\']], \'complete\':True,\'supabaseSourceGETs\':0}\n    atomic(out/\'baseline-receipt.json\',json.dumps(receipt,indent=2).encode())\n    return m\n\ndef main():\n    p=argparse.ArgumentParser(); p.add_argument(\'--output\',required=True); p.add_argument(\'--mounted\'); p.add_argument(\'--manifest\'); p.add_argument(\'--packages\')\n    a=p.parse_args(); prepare_baseline(a.output,mounted=a.mounted,local_manifest=a.manifest,package_dir=a.packages,drive=None if a.mounted or a.manifest else Drive())\nif __name__==\'__main__\': main()\n', 'provider.py': 'TARGET_COUNT = FLOW_COUNT = 1\n"""Provider functions copied from the SHA-pinned existing KR v3.3.3 runtime.\nOriginal bundle SHA256: 91d4992be9a2d06916ee2e425cb43d210350eed5e73a83429dbb4d041c8b7aba\nOnly reviewed definitions/constants are vendored. No runtime archive execution, publisher or pickle.\n"""\n# 패키지 설치는 첫 번째 Colab bootstrap 셀에서 완료합니다.\n\nimport os\nimport re\nimport sys\nimport time\nimport random\nimport json\nimport warnings\nimport threading\nimport math\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\n\nimport requests\nimport pandas as pd\nfrom bs4 import BeautifulSoup\nfrom tqdm.auto import tqdm\nfrom IPython.display import display, FileLink\n\nfrom urllib.parse import urlparse as _bootstrap_urlparse\n\npass  # 공식 API 경로만 사용\n\npd.set_option("display.max_columns", 200)\nwarnings.filterwarnings("default")\n\n# Python 3.13 + jupyter_client 자체의 utcnow 경고가 tqdm 화면을 반복해서 밀어내는\n# 현상만 숨깁니다. 수집 코드에서 발생하는 다른 경고는 그대로 표시합니다.\nwarnings.filterwarnings(\n    "ignore",\n    message=r"datetime\\.datetime\\.utcnow\\(\\) is deprecated.*",\n    category=DeprecationWarning,\n    module=r"jupyter_client\\.session",\n)\n# Google Drive 마운트는 bootstrap 셀에서 한 번만 처리합니다.\nIN_COLAB = "google.colab" in sys.modules\n\nimport os, json, time, math, re\nfrom pathlib import Path\nfrom datetime import datetime\nfrom zoneinfo import ZoneInfo\nimport pandas as pd\nimport requests\n\nKRX_OFFICIAL_SERVICES = {\'KOSPI\': \'sto/stk_bydd_trd\', \'KOSDAQ\': \'sto/ksq_bydd_trd\', \'ETF\': \'etp/etf_bydd_trd\'}\nKRX_OFFICIAL_BASE_URL = \'https://data-dbg.krx.co.kr/svc/apis/\'\nNUMBERS = {\'TDD_OPNPRC\':\'open\',\'TDD_HGPRC\':\'high\',\'TDD_LWPRC\':\'low\',\'TDD_CLSPRC\':\'close\',\n           \'ACC_TRDVOL\':\'volume\',\'ACC_TRDVAL\':\'tradingValue\',\'MKTCAP\':\'marketCap\',\'LIST_SHRS\':\'listedShares\'}\nETF_NUMBERS = {\'NAV\':\'etfNav\',\'MKTCAP\':\'etfMarketCap\',\'INVSTASST_NETASST_TOTAMT\':\'etfNetAssetTotalAmount\',\n               \'LIST_SHRS\':\'etfListedUnits\',\'OBJ_STKPRC_IDX\':\'etfUnderlyingIndexClose\'}\nENRICH_FIELDS = [\'marketCap\',\'krxMarketCap\',\'listedShares\',\'krxListedShares\',\n                *ETF_NUMBERS.values(),\'etfUnderlyingIndexName\']\n\nclass OfficialAPIError(RuntimeError): pass\n\ndef official_atomic_json(path, data):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    temp=path.with_suffix(\'.tmp\');temp.write_text(json.dumps(data,ensure_ascii=False),encoding=\'utf-8\');temp.replace(path)\n\ndef official_numeric(value):\n    if value is None or str(value).strip() in (\'\',\'-\',\'--\'): return None\n    try: n=float(str(value).replace(\',\',\'\'))\n    except ValueError: raise OfficialAPIError(\'숫자 응답 형식 오류\') from None\n    if not math.isfinite(n) or n<0: raise OfficialAPIError(\'음수 또는 비유한 숫자 응답\')\n    return n\n\ndef official_normalize(rows, market, day):\n    result=[];seen=set()\n    for r in rows:\n        if not isinstance(r,dict): raise OfficialAPIError(\'행 형식 오류\')\n        rawday=str(r.get(\'BAS_DD\',\'\')).replace(\'-\',\'\')\n        if rawday!=day: raise OfficialAPIError(\'요청일과 응답일 불일치\')\n        symbol=str(r.get(\'ISU_CD\',\'\')).strip().upper()\n        if symbol.isdigit():symbol=symbol.zfill(6)\n        if not re.fullmatch(\'[0-9A-Z]{6}\',symbol):raise OfficialAPIError(\'6자리 종목코드 형식 불일치\')\n        if symbol in seen:raise OfficialAPIError(\'종목·날짜 중복 응답\')\n        seen.add(symbol)\n        required={\'TDD_CLSPRC\',\'MKTCAP\',\'ACC_TRDVAL\'} | (set(ETF_NUMBERS) if market==\'ETF\' else set())\n        if not required.issubset(r):raise OfficialAPIError(\'공식 명세 필수 응답 열 누락\')\n        row={\'symbol\':symbol,\'date\':datetime.strptime(day,\'%Y%m%d\').strftime(\'%Y-%m-%d\'),\n             \'name\':r.get(\'ISU_NM\',\'\'),\'market\':market,\'securityType\':\'ETF\' if market==\'ETF\' else \'STOCK\'}\n        row.update({target:official_numeric(r.get(src)) for src,target in NUMBERS.items()})\n        for col in [\'marketCap\',\'listedShares\']:\n            if row[col]==0:row[col]=None\n        row[\'krxMarketCap\']=row[\'marketCap\'];row[\'krxListedShares\']=row[\'listedShares\']\n        row[\'marketCapSource\']=\'KRX_ETF\' if market==\'ETF\' else \'KRX_STOCK\'\n        row[\'priceSource\']=\'KRX_OPENAPI_UNADJUSTED\'\n        if market==\'ETF\':\n            row.update({target:official_numeric(r.get(src)) for src,target in ETF_NUMBERS.items()})\n            for target in ETF_NUMBERS.values():\n                if row[target]==0:row[target]=None\n            row[\'etfUnderlyingIndexName\']=r.get(\'IDX_IND_NM\') or None\n        result.append(row)\n    return pd.DataFrame(result)\n\nclass OfficialClient:\n    def __init__(self,key,root,budget=9000,refresh=False):\n        if not key:raise OfficialAPIError(\'승인받은 KRX_OPENAPI_KEY가 필요합니다. KRX_ID/KRX_PW와 다릅니다.\')\n        self.key=key;self.root=Path(root);self.budget=int(budget);self.refresh=refresh\n        if not 1<=self.budget<=9000:raise ValueError(\'요청 예산은 1~9000입니다.\')\n        self.session=requests.Session();self.last=0\n    def fetch(self,market,day):\n        if market not in KRX_OFFICIAL_SERVICES:raise ValueError(\'지원하지 않는 시장\')\n        cache=self.root/\'cache\'/market/(day+\'.json\')\n        if cache.exists() and not self.refresh:\n            rows=json.loads(cache.read_text(encoding=\'utf-8\'));official_normalize(rows,market,day);return rows\n        today=datetime.now(ZoneInfo(\'Asia/Seoul\')).strftime(\'%Y%m%d\')\n        counter=self.root/\'usage\'/(today+\'.json\')\n        used=json.loads(counter.read_text()).get(\'attempts\',0) if counter.exists() else 0\n        if used>=self.budget:raise OfficialAPIError(\'로컬 일일 요청 예산 도달. 캐시는 보존됩니다.\')\n        time.sleep(max(0,.4-(time.monotonic()-self.last)))\n        official_atomic_json(counter,{\'attempts\':used+1});self.last=time.monotonic()\n        try:\n            resp=self.session.get(KRX_OFFICIAL_BASE_URL+KRX_OFFICIAL_SERVICES[market],params={\'basDd\':day},\n                                  headers={\'AUTH_KEY\':self.key},timeout=(10,45),allow_redirects=False)\n        except requests.RequestException:raise OfficialAPIError(\'네트워크 오류. 자동 재시도하지 않습니다.\') from None\n        if resp.status_code!=200:raise OfficialAPIError(f\'공식 API HTTP {resp.status_code}: 인증키·서비스 승인·한도를 확인하세요.\')\n        try:body=resp.json()\n        except ValueError:raise OfficialAPIError(\'공식 API 비JSON 응답. 자동 재시도하지 않습니다.\') from None\n        if not isinstance(body,dict) or not isinstance(body.get(\'OutBlock_1\'),list):\n            raise OfficialAPIError(\'OutBlock_1 응답 없음. 키·서비스 승인 상태를 확인하세요.\')\n        rows=body[\'OutBlock_1\'];official_normalize(rows,market,day)\n        # Empty days may be holidays or not yet published; never cache as success.\n        if rows:official_atomic_json(cache,rows)\n        return rows\n\n# Official API adapter: shared daily requests, exact date joins, bounded disk index.\nimport sqlite3\nimport tempfile\n_OFFICIAL_DB = None\n_OFFICIAL_LOADED = set()\n_OFFICIAL_LOCK = threading.RLock()\n_OFFICIAL_CLIENT = None\n_OFFICIAL_FAILURE = None\n_OFFICIAL_QA = []\n\nclass KRXSchemaError(ValueError): pass\nclass KRXServiceUnavailable(RuntimeError): pass\n\ndef official_client():\n    global _OFFICIAL_CLIENT\n    if _OFFICIAL_CLIENT is None:\n        root = Path(os.environ.get(\'CLOUDTREND_STATE_ROOT\', \'/content/drive/MyDrive/CloudTrend/toss_krx_integrated\')) / \'official_api_v1\'\n        _OFFICIAL_CLIENT = OfficialClient(os.environ.get(\'KRX_OPENAPI_KEY\'), root, budget=9000, refresh=globals().get(\'REFRESH_CACHE\', False))\n    return _OFFICIAL_CLIENT\n\ndef official_db():\n    global _OFFICIAL_DB\n    if _OFFICIAL_DB is None:\n        handle = tempfile.NamedTemporaryFile(prefix=\'cloudtrend_official_\', suffix=\'.sqlite\', delete=False)\n        handle.close()\n        _OFFICIAL_DB = sqlite3.connect(handle.name, check_same_thread=False)\n        _OFFICIAL_DB.execute(\'CREATE TABLE rows (market TEXT, date TEXT, symbol TEXT, payload TEXT, PRIMARY KEY(market,date,symbol))\')\n        _OFFICIAL_DB.execute(\'CREATE INDEX symbol_dates ON rows(symbol,date)\')\n    return _OFFICIAL_DB\n\ndef official_day(market, day, ensure_only=False):\n    global _OFFICIAL_FAILURE\n    day = pd.Timestamp(day).strftime(\'%Y%m%d\')\n    iso = pd.Timestamp(day).strftime(\'%Y-%m-%d\')\n    with _OFFICIAL_LOCK:\n        if _OFFICIAL_FAILURE:\n            raise KRXServiceUnavailable(_OFFICIAL_FAILURE)\n        db = official_db()\n        key = (market, day)\n        if key not in _OFFICIAL_LOADED:\n            try:\n                raw = official_client().fetch(market, day)\n                frame = official_normalize(raw, market, day)\n            except OfficialAPIError as exc:\n                _OFFICIAL_FAILURE = str(exc)\n                raise KRXServiceUnavailable(_OFFICIAL_FAILURE) from None\n            # ETF master-only rows on weekends are NOT trading observations.\n            if not frame.empty:\n                frame = frame[pd.to_numeric(frame[\'close\'], errors=\'coerce\').gt(0) & pd.to_numeric(frame[\'marketCap\'], errors=\'coerce\').gt(0)].copy()\n            status = \'received\' if len(frame) else (\'reference_only_no_prices\' if raw else \'empty_unconfirmed\')\n            _OFFICIAL_QA.append({\'date\':iso,\'market\':market,\'rawRows\':len(raw),\'usableRows\':len(frame),\'status\':status})\n            if \'OUTPUT_ROOT\' in globals() and (len(_OFFICIAL_QA) == 1 or len(_OFFICIAL_QA) % 25 == 0):\n                pd.DataFrame(_OFFICIAL_QA).to_csv(OUTPUT_ROOT/\'official_api_coverage.csv\',index=False,encoding=\'utf-8-sig\')\n            for row in frame.to_dict(\'records\'):\n                db.execute(\'INSERT OR REPLACE INTO rows VALUES (?,?,?,?)\',(market,iso,row[\'symbol\'],json.dumps(row,ensure_ascii=False)))\n            db.commit()\n            _OFFICIAL_LOADED.add(key)\n            if len(_OFFICIAL_QA)%25==0:\n                print(f\'KRX 공식 API 날짜×시장 {len(_OFFICIAL_QA):,}개 확인 (공유 캐시 사용)\')\n        if ensure_only:\n            return None\n        rows = db.execute(\'SELECT payload FROM rows WHERE market=? AND date=?\',(market,iso)).fetchall()\n        return pd.DataFrame([json.loads(r[0]) for r in rows])\n\ndef official_symbol_frame(symbol, start, end, etf=False):\n    # Use actual Toss candle dates: no holiday rows and no fabricated dates.\n    candles = cached_frame(\'toss_candles\',symbol,lambda:get_candles_frame(symbol,TARGET_COUNT,False))\n    dates = sorted(set(candles.loc[candles.date.between(pd.Timestamp(start).strftime(\'%Y-%m-%d\'),pd.Timestamp(end).strftime(\'%Y-%m-%d\')),\'date\']))\n    markets = [\'ETF\'] if etf else [\'KOSPI\',\'KOSDAQ\'] # both for historical market transfers\n    for day in dates:\n        for market in markets:\n            official_day(market,day,ensure_only=True)\n    with _OFFICIAL_LOCK:\n        rows=official_db().execute(\'SELECT payload FROM rows WHERE symbol=? AND date BETWEEN ? AND ? ORDER BY date\',\n            (symbol,pd.Timestamp(start).strftime(\'%Y-%m-%d\'),pd.Timestamp(end).strftime(\'%Y-%m-%d\'))).fetchall()\n    f=pd.DataFrame([json.loads(r[0]) for r in rows])\n    if f.empty:return pd.DataFrame(columns=[\'date\'])\n    if f.date.duplicated().any():raise KRXSchemaError(\'동일 종목·날짜의 시장 중복\')\n    mapping = {\'tradingValue\':\'etfTradingValue\' if etf else \'krxTradingValue\', \'volume\':\'krxVolume\'}\n    cols=[\'date\',\'tradingValue\'] + ([\'etfNav\',\'etfMarketCap\',\'etfNetAssetTotalAmount\',\'etfListedUnits\',\'etfUnderlyingIndexClose\',\'etfUnderlyingIndexName\'] if etf else [\'krxMarketCap\',\'krxListedShares\',\'volume\'])\n    return f[cols].rename(columns=mapping)\n\ndef get_krx_market_frame(symbol,start_yyyymmdd,end_yyyymmdd):\n    return official_symbol_frame(symbol,start_yyyymmdd,end_yyyymmdd)\n\ndef get_krx_etf_reference_frame(symbol,start_yyyymmdd,end_yyyymmdd):\n    return official_symbol_frame(symbol,start_yyyymmdd,end_yyyymmdd,True)\n\ndef get_krx_etf_investor_value_frame(symbol,start_yyyymmdd,end_yyyymmdd):\n    # Official service has no individual-stock/ETF investor amount history.\n    return pd.DataFrame(columns=[\'date\'])\n\ndef get_krx_market_cap_market_day(trade_date,market):\n    f=official_day(market,trade_date)\n    cols=[\'date\',\'symbol\',\'krxLatestMarketCap\',\'krxLatestListedShares\']\n    if f.empty:return pd.DataFrame(columns=cols)\n    return f[[\'date\',\'symbol\',\'marketCap\',\'listedShares\']].rename(columns={\'marketCap\':\'krxLatestMarketCap\',\'listedShares\':\'krxLatestListedShares\'})\n\ndef apply_screening_krx_foreign_batch(frame):\n    print(\'공식 API 미제공: 개별 종목 외국인 순매수금액. Toss 수량 및 명시된 기존 추정금액을 유지합니다.\')\n    return frame.copy()\n\ndef diagnose_krx_connection():\n    result={\'provider\':\'KRX_OPENAPI\',\'ok\':False,\'markets\':{}}\n    for market in ([\'KOSPI\',\'KOSDAQ\'] if UNIVERSE_MODE==\'STOCK_ONLY\' else [\'ETF\'] if UNIVERSE_MODE==\'ETF_ONLY\' else [\'KOSPI\',\'KOSDAQ\',\'ETF\']):\n        for day in pd.date_range(end=COLLECTION_CUTOFF_DATE,periods=10)[::-1]:\n            if day.weekday()>=5:continue\n            f=official_day(market,day)\n            if not f.empty:\n                result[\'markets\'][market]={\'date\':day.strftime(\'%Y-%m-%d\'),\'rows\':len(f)}\n                break\n        if market not in result[\'markets\']:\n            raise KRXServiceUnavailable(f\'{market}: 최근 조회일에 유효 가격·시총 자료가 없습니다. 공식 API 승인·공표일을 확인하세요.\')\n    result[\'ok\']=True\n    print(json.dumps(result,ensure_ascii=False,indent=2))\n    return result\n\ndef ensure_krx_connection():\n    diagnose_krx_connection()\n\ndef official_current_etfs():\n    end = globals().get(\'COLLECTION_CUTOFF_DATE\') or pd.Timestamp.now(tz=\'Asia/Seoul\').date().isoformat()\n    for day in pd.date_range(end=end,periods=10)[::-1]:\n        if day.weekday()>=5:continue\n        f=official_day(\'ETF\',day)\n        if not f.empty:\n            print(f\'공식 API ETF 목록: {day.date()} / {len(f):,}종\')\n            return f.symbol.tolist()\n    raise KRXServiceUnavailable(\'공식 API ETF 목록의 유효 거래일을 찾지 못했습니다.\')\n\nerrors = []\n\n_errors_lock = threading.Lock()\n\ndef add_error(stage, symbol=None, endpoint=None, error=None):\n    with _errors_lock:\n        errors.append({\n            "time": datetime.now(timezone.utc).isoformat(timespec="seconds"),\n            "stage": stage,\n            "symbol": symbol,\n            "endpoint": endpoint,\n            "error": str(error)[:2000] if error is not None else None,\n        })\n\ndef normalize_symbol(value):\n    if value is None:\n        return None\n    s = str(value).strip().upper()\n    return s or None\n\ndef first_text(*values):\n    for v in values:\n        if v is None:\n            continue\n        s = str(v).strip()\n        if s and s.lower() not in {"none", "nan", "null"}:\n            return s\n    return None\n\ndef to_number(value):\n    if value is None:\n        return None\n    if isinstance(value, bool):\n        return float(value)\n    s = str(value).replace(",", "").replace("₩", "").replace("%", "").strip()\n    if not s or s in {"-", "--"} or s.lower() in {"none", "null", "nan"}:\n        return None\n    try:\n        return float(s)\n    except Exception:\n        return None\n\ndef first_number(*values):\n    for v in values:\n        n = to_number(v)\n        if n is not None:\n            return n\n    return None\n\ndef unwrap_payload(payload):\n    if isinstance(payload, dict) and "result" in payload:\n        return payload["result"]\n    return payload\n\ndef extract_list(payload, keys=("records", "candles", "stocks", "items", "results")):\n    obj = unwrap_payload(payload)\n    if isinstance(obj, list):\n        return obj\n    if isinstance(obj, dict):\n        for key in keys:\n            value = obj.get(key)\n            if isinstance(value, list):\n                return value\n    return []\n\ndef extract_cursor(payload, keys=("nextUntil", "nextBefore", "nextCursor", "cursor")):\n    obj = unwrap_payload(payload)\n    if isinstance(obj, dict):\n        for key in keys:\n            value = obj.get(key)\n            if value not in (None, ""):\n                return str(value)\n    if isinstance(payload, dict):\n        for key in keys:\n            value = payload.get(key)\n            if value not in (None, ""):\n                return str(value)\n    return None\n\ndef to_trade_date(value):\n    if value is None:\n        return None\n    s = str(value).strip()\n    if not s:\n        return None\n    try:\n        ts = pd.Timestamp(s)\n        # Toss 공식 timestamp는 +09:00 offset 포함. naive면 날짜 자체를 유지한다.\n        if ts.tzinfo is not None:\n            ts = ts.tz_convert("Asia/Seoul")\n        return ts.strftime("%Y-%m-%d")\n    except Exception:\n        digits = re.sub(r"\\D", "", s)\n        if len(digits) >= 8:\n            return f"{digits[:4]}-{digits[4:6]}-{digits[6:8]}"\n        return None\n\ndef split_batches(values, size):\n    for i in range(0, len(values), size):\n        yield values[i:i+size]\n\ndef normalize_market(value):\n    text = first_text(value)\n    if text is None:\n        return None\n    upper = text.upper()\n    if "KOSDAQ" in upper:\n        return "KOSDAQ"\n    if "KOSPI" in upper:\n        return "KOSPI"\n    if "KONEX" in upper:\n        return "KONEX"\n    return text\n\ndef safe_filename(text):\n    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text))\n\n_REFRESHED_STAGE_KEYS = set()\n\ndef validate_toss_openapi(strict=True):\n    required_paths = {\n        "/api/v1/candles",\n        "/api/v1/stocks/{symbol}/investor-trading",\n        "/api/v1/stocks/{symbol}/program-trades",\n        "/api/v1/stocks/{symbol}/short-selling",\n        "/api/v1/stocks/{symbol}/credit-trades",\n        "/api/v1/stocks/{symbol}/securities-lending",\n    }\n    try:\n        r = requests.get(OPENAPI_SPEC_URL, timeout=20)\n        r.raise_for_status()\n        spec = r.json()\n        version = spec.get("info", {}).get("version")\n        paths = set(spec.get("paths", {}))\n        missing = sorted(required_paths - paths)\n        print(f"Toss OpenAPI server schema version: {version}")\n        if missing:\n            msg = "필수 endpoint가 현재 OpenAPI에서 사라졌습니다: " + ", ".join(missing)\n            if strict:\n                raise RuntimeError(msg)\n            warnings.warn(msg)\n        return version\n    except Exception as e:\n        if strict:\n            raise\n        warnings.warn(f"OpenAPI schema 확인 실패: {e}")\n        return None\n\nclass PacedRateLimiter:\n    """여러 worker가 같은 공식 Rate Limits Group 한도를 넘지 않도록 시작 시각을 예약합니다."""\n\n    def __init__(self, minimum_gap):\n        self.minimum_gap = float(minimum_gap)\n        self.next_at = 0.0\n        self.lock = threading.Lock()\n\n    def acquire(self, requested_gap=None):\n        gap = max(self.minimum_gap, float(requested_gap or 0.0))\n        with self.lock:\n            now = time.monotonic()\n            slot = max(now, self.next_at)\n            self.next_at = slot + gap\n        delay = slot - now\n        if delay > 0:\n            time.sleep(delay)\n\n    def pause(self, seconds):\n        with self.lock:\n            self.next_at = max(self.next_at, time.monotonic() + max(0.0, seconds))\n\nclass TossClient:\n    """Thread-safe OAuth client with per-rate-group pacing and pooled connections."""\n\n    def __init__(self):\n        self.timeout = (TOSS_CONNECT_TIMEOUT, TOSS_READ_TIMEOUT)\n        self.max_retries = TOSS_MAX_RETRIES\n        self._local = threading.local()\n        self._token_lock = threading.Lock()\n        self._token = None\n        self._token_generation = 0\n        self._metrics_lock = threading.Lock()\n        self._request_count = 0\n        self._retry_count = 0\n        self._rate_limit_count = 0\n        self._observed_limits = {}\n        self._limiters = {\n            "MARKET_DATA_CHART": PacedRateLimiter(TOSS_CHART_GAP),\n            "MARKET_INDICATOR_CHART": PacedRateLimiter(TOSS_INDICATOR_CHART_GAP),\n            "STOCK_TRADING_TREND": PacedRateLimiter(TOSS_TREND_GAP),\n            "STOCK": PacedRateLimiter(TOSS_STOCK_GAP),\n            "MARKET_INDICATOR": PacedRateLimiter(TOSS_INDICATOR_GAP),\n        }\n        self.issue_token()\n\n    def _session(self):\n        session = getattr(self._local, "session", None)\n        if session is None:\n            session = requests.Session()\n            adapter = requests.adapters.HTTPAdapter(\n                pool_connections=TOSS_WORKERS + 2,\n                pool_maxsize=TOSS_WORKERS + 2,\n                max_retries=0,\n                pool_block=True,\n            )\n            session.mount("https://", adapter)\n            session.headers.update({"Accept": "application/json"})\n            self._local.session = session\n        return session\n\n    @staticmethod\n    def _group_for(endpoint):\n        if "/market-indicators/" in endpoint and endpoint.endswith("/candles"):\n            return "MARKET_INDICATOR_CHART"\n        if endpoint.startswith("/api/v1/market-indicators/"):\n            return "MARKET_INDICATOR"\n        if endpoint == "/api/v1/candles":\n            return "MARKET_DATA_CHART"\n        if endpoint.startswith("/api/v1/stocks/") and endpoint.rsplit("/", 1)[-1] in {\n            "investor-trading", "program-trades", "short-selling",\n            "credit-trades", "securities-lending",\n        }:\n            return "STOCK_TRADING_TREND"\n        return "STOCK"\n\n    def issue_token(self, observed_generation=None):\n        # client당 유효 token은 하나뿐이므로 refresh를 단일화합니다.\n        with self._token_lock:\n            if observed_generation is not None and observed_generation != self._token_generation:\n                return\n            response = requests.post(\n                BASE + "/oauth2/token",\n                headers={"Content-Type": "application/x-www-form-urlencoded"},\n                data={\n                    "grant_type": "client_credentials",\n                    "client_id": CLIENT_ID,\n                    "client_secret": CLIENT_SECRET,\n                },\n                timeout=self.timeout,\n            )\n            if not response.ok:\n                raise RuntimeError(\n                    f"Toss 토큰 발급 실패 HTTP {response.status_code}"\n                )\n            payload = response.json()\n            token = payload.get("access_token")\n            if not token:\n                obj = unwrap_payload(payload)\n                if isinstance(obj, dict):\n                    token = obj.get("access_token")\n            if not token:\n                raise RuntimeError("Toss token 응답에 access_token이 없습니다.")\n            self._token = token\n            self._token_generation += 1\n\n    def _token_snapshot(self):\n        with self._token_lock:\n            return self._token, self._token_generation\n\n    def _record_response(self, group, response):\n        limit = response.headers.get("X-RateLimit-Limit")\n        with self._metrics_lock:\n            self._request_count += 1\n            if response.status_code == 429:\n                self._rate_limit_count += 1\n            if limit:\n                self._observed_limits[group] = limit\n\n    def metrics(self):\n        with self._metrics_lock:\n            return {\n                "requests": self._request_count,\n                "retries": self._retry_count,\n                "http_429": self._rate_limit_count,\n                "observed_limits": dict(self._observed_limits),\n            }\n\n    def get(self, endpoint, minimum_gap=None, **params):\n        group = self._group_for(endpoint)\n        limiter = self._limiters[group]\n        last_error = None\n\n        for attempt in range(self.max_retries):\n            if COLLECTION_STOP_EVENT.is_set():\n                raise InterruptedError("수집 중단 요청")\n            if attempt > 0:\n                with self._metrics_lock:\n                    self._retry_count += 1\n            limiter.acquire(minimum_gap)\n            token, generation = self._token_snapshot()\n            try:\n                response = self._session().get(\n                    BASE + endpoint,\n                    params=params,\n                    headers={"Authorization": f"Bearer {token}"},\n                    timeout=self.timeout,\n                )\n            except requests.RequestException as e:\n                last_error = e\n                time.sleep(min(6.0, 0.4 * (2 ** attempt)) + random.random() * 0.15)\n                continue\n\n            self._record_response(group, response)\n\n            if response.status_code == 401:\n                self.issue_token(observed_generation=generation)\n                continue\n\n            if response.status_code == 429:\n                retry_after = response.headers.get("Retry-After")\n                reset_after = response.headers.get("X-RateLimit-Reset")\n                try:\n                    delay = float(retry_after or reset_after or (0.75 * (attempt + 1)))\n                except Exception:\n                    delay = 0.75 * (attempt + 1)\n                last_error = RuntimeError(f"HTTP 429, {delay:.2f}초 후 재시도")\n                limiter.pause(max(0.5, delay))\n                continue\n\n            if 500 <= response.status_code < 600:\n                last_error = RuntimeError(f"HTTP {response.status_code}")\n                time.sleep(min(6.0, 0.4 * (2 ** attempt)) + random.random() * 0.15)\n                continue\n\n            if not response.ok:\n                raise RuntimeError(\n                    f"Toss API 오류 {endpoint} HTTP {response.status_code}"\n                )\n\n            return response.json()\n\n        raise RuntimeError(f"Toss API 반복 실패 {endpoint}: {last_error}")\n\ndef get_market_master():\n    output = {}\n    for market in ["KOSPI", "KOSDAQ"]:\n        try:\n            payload = client.get(\n                "/api/v1/stocks/all",\n                minimum_gap=1.05,\n                market=market,\n                status="ACTIVE",\n            )\n            for record in extract_list(payload, ("stocks", "items", "records", "results")):\n                if not isinstance(record, dict):\n                    continue\n                symbol = normalize_symbol(\n                    first_text(\n                        record.get("symbol"), record.get("code"),\n                        record.get("ticker"), record.get("shortCode"),\n                    )\n                )\n                if not symbol:\n                    continue\n                output[symbol] = {\n                    **record,\n                    "symbol": symbol,\n                    "name": first_text(\n                        record.get("name"), record.get("displayName"),\n                        record.get("koreanName"), record.get("securityName"), symbol,\n                    ),\n                    "market": market,\n                }\n        except Exception as e:\n            add_error("market_master", market, "/api/v1/stocks/all", e)\n    return output\n\ndef get_stock_details(symbols):\n    output = {}\n    for batch in tqdm(list(split_batches(symbols, 200)), desc="Toss 종목 상세"):\n        try:\n            payload = client.get(\n                "/api/v1/stocks",\n                minimum_gap=0.35,\n                symbols=",".join(batch),\n            )\n            for record in extract_list(payload, ("stocks", "items", "records", "results")):\n                if not isinstance(record, dict):\n                    continue\n                symbol = normalize_symbol(\n                    first_text(\n                        record.get("symbol"), record.get("code"),\n                        record.get("ticker"), record.get("shortCode"),\n                    )\n                )\n                if symbol:\n                    output[symbol] = record\n        except Exception as e:\n            add_error("stock_details", ",".join(batch[:10]), "/api/v1/stocks", e)\n    return output\n\ndef get_candles_frame(symbol, target_count=TARGET_COUNT, is_index=False):\n    endpoint = (\n        f"/api/v1/market-indicators/{symbol}/candles"\n        if is_index else "/api/v1/candles"\n    )\n    output = {}\n    # 최초 요청부터 지정 기준일 이전 구간으로 이동합니다.\n    # Toss before는 inclusive ISO 8601이므로 전날 23:59:59를 사용합니다.\n    before = COLLECTION_CUTOFF_BEFORE\n    seen = set()\n    max_pages = max(20, (target_count // 200) + 5)\n\n    for _ in range(max_pages):\n        if COLLECTION_STOP_EVENT.is_set():\n            raise InterruptedError("수집 중단 요청")\n        params = {"interval": "1d", "count": min(200, target_count)}\n        if not is_index:\n            params.update({"symbol": symbol, "adjusted": "true"})\n        if before is not None:\n            params["before"] = before\n\n        payload = client.get(endpoint, minimum_gap=TOSS_CHART_GAP, **params)\n        records = extract_list(payload, ("candles", "records", "items", "results"))\n        if not records:\n            break\n\n        for rec in records:\n            if not isinstance(rec, dict):\n                continue\n            ts = first_text(rec.get("timestamp"), rec.get("date"), rec.get("tradeDate"))\n            date = to_trade_date(ts)\n            if not date:\n                continue\n            if COLLECTION_CUTOFF_DATE is not None and date > COLLECTION_CUTOFF_DATE:\n                continue\n            output[date] = {\n                "date": date,\n                "open": first_number(rec.get("openPrice")),\n                "high": first_number(rec.get("highPrice")),\n                "low": first_number(rec.get("lowPrice")),\n                "close": first_number(rec.get("closePrice")),\n                "volume": first_number(rec.get("volume")),\n            }\n\n        if len(output) >= target_count:\n            break\n        cursor = extract_cursor(payload, ("nextBefore",))\n        if not cursor or cursor in seen:\n            break\n        seen.add(cursor)\n        before = cursor\n\n    if not output:\n        return pd.DataFrame()\n    df = pd.DataFrame(output.values()).sort_values("date").tail(target_count).reset_index(drop=True)\n    return df\n\ndef fetch_trend_frame(symbol, suffix, parser, target_count=FLOW_COUNT):\n    endpoint = f"/api/v1/stocks/{symbol}/{suffix}"\n    output = {}\n    # Toss until은 inclusive YYYY-MM-DD이므로 기준일 전날을 최초 상한으로 사용합니다.\n    until = COLLECTION_CUTOFF_DATE\n    seen = set()\n    max_pages = max(20, (target_count // 100) + 5)\n\n    for _ in range(max_pages):\n        if COLLECTION_STOP_EVENT.is_set():\n            raise InterruptedError("수집 중단 요청")\n        params = {"count": min(100, target_count)}\n        if until is not None:\n            params["until"] = until\n\n        payload = client.get(endpoint, minimum_gap=TOSS_TREND_GAP, **params)\n        records = extract_list(payload, ("records",))\n        if not records:\n            break\n\n        for rec in records:\n            if not isinstance(rec, dict):\n                continue\n            date = to_trade_date(rec.get("date"))\n            if not date:\n                continue\n            if COLLECTION_CUTOFF_DATE is not None and date > COLLECTION_CUTOFF_DATE:\n                continue\n            row = parser(rec)\n            row["date"] = date\n            output[date] = row\n\n        if len(output) >= target_count:\n            break\n        cursor = extract_cursor(payload, ("nextUntil",))\n        if not cursor or cursor in seen:\n            break\n        seen.add(cursor)\n        until = cursor\n\n    if not output:\n        return pd.DataFrame()\n    return (\n        pd.DataFrame(output.values())\n        .sort_values("date")\n        .tail(target_count)\n        .reset_index(drop=True)\n    )\n\ndef parse_short_record(record):\n    return {\n        **{key: first_number(record.get(key)) for key in (\n            \'shortSellingVolume\', \'shortSellingAmount\',\n            \'shortSellingVolumeRate\', \'shortSellingAmountRate\')},\n        \'shortUpdatedAt\': first_text(record.get(\'updatedAt\')),\n    }\n\ndef parse_lending_record(record):\n    fields = {\'executionQuantity\':\'lendingExecutionQuantity\',\n              \'repaymentQuantity\':\'lendingRepaymentQuantity\',\n              \'balanceQuantity\':\'lendingBalanceQuantity\',\n              \'balanceAmount\':\'lendingBalanceAmount\'}\n    return {**{dest:first_number(record.get(src)) for src,dest in fields.items()},\n            \'lendingUpdatedAt\':first_text(record.get(\'updatedAt\'))}\n\ndef _volume_triplet(obj, prefix):\n    obj = obj if isinstance(obj, dict) else {}\n    return {\n        f"{prefix}BuyVolume": first_number(obj.get("buyVolume")),\n        f"{prefix}SellVolume": first_number(obj.get("sellVolume")),\n        f"{prefix}NetBuyVolume": first_number(obj.get("netBuyVolume")),\n    }\n\ndef parse_investor_record(record):\n    individual = record.get("individual")\n    foreigner = record.get("foreigner")\n    institution = record.get("institution")\n    other = record.get("otherCorporation")\n    institution = institution if isinstance(institution, dict) else {}\n    breakdown = institution.get("breakdown")\n    breakdown = breakdown if isinstance(breakdown, dict) else {}\n    holding = record.get("foreignerHolding")\n    holding = holding if isinstance(holding, dict) else {}\n    cfd = record.get("cfd")\n    cfd = cfd if isinstance(cfd, dict) else {}\n\n    out = {}\n    out.update(_volume_triplet(individual, "individual"))\n    out.update(_volume_triplet(foreigner, "foreign"))\n    out.update(_volume_triplet(institution, "institution"))\n    out.update(_volume_triplet(other, "otherCorporation"))\n\n    detail_map = {\n        "financialInvestment": "financialInvestment",\n        "insurance": "insurance",\n        "trust": "trust",\n        "privateEquityFund": "privateEquityFund",\n        "bank": "bank",\n        "otherFinancialInstitution": "otherFinancialInstitution",\n        "pensionFund": "pensionFund",\n    }\n    for api_key, prefix in detail_map.items():\n        part = breakdown.get(api_key)\n        part = part if isinstance(part, dict) else {}\n        out[f"{prefix}NetBuyVolume"] = first_number(part.get("netBuyVolume"))\n\n    rate = first_number(holding.get("holdingRate"))\n    out.update({\n        "foreignHoldingQuantity": first_number(holding.get("holdingQuantity")),\n        "foreignHoldingLimitQuantity": first_number(holding.get("limitQuantity")),\n        "foreignHoldingRate": rate,\n        "foreignHoldingRatePct": rate * 100 if rate is not None else None,\n        "cfdBuyBalanceQuantity": first_number(cfd.get("buyBalanceQuantity")),\n        "cfdBuyBalanceRate": first_number(cfd.get("buyBalanceRate")),\n        "cfdSellBalanceQuantity": first_number(cfd.get("sellBalanceQuantity")),\n        "cfdSellBalanceRate": first_number(cfd.get("sellBalanceRate")),\n        "investorUpdatedAt": first_text(record.get("updatedAt")),\n    })\n    return out\n\ndef get_market_investor_flow_frame(symbol, target_count=FLOW_COUNT):\n    endpoint = f"/api/v1/market-indicators/{symbol}/investor-trading"\n    output = {}\n    # 지수 수급도 동일하게 기준일 이전으로 시작합니다.\n    until = COLLECTION_CUTOFF_DATE\n    seen = set()\n    max_pages = max(20, (target_count // 100) + 5)\n\n    for _ in range(max_pages):\n        if COLLECTION_STOP_EVENT.is_set():\n            raise InterruptedError("수집 중단 요청")\n        params = {"interval": "1d", "count": min(100, target_count)}\n        if until:\n            params["until"] = until\n        payload = client.get(endpoint, minimum_gap=TOSS_TREND_GAP, **params)\n        records = extract_list(payload, ("records",))\n        if not records:\n            break\n\n        for rec in records:\n            if not isinstance(rec, dict):\n                continue\n            date = to_trade_date(rec.get("date"))\n            if not date:\n                continue\n            if COLLECTION_CUTOFF_DATE is not None and date > COLLECTION_CUTOFF_DATE:\n                continue\n\n            def net_amount(key):\n                obj = rec.get(key)\n                obj = obj if isinstance(obj, dict) else {}\n                buy = first_number(obj.get("buyAmount"))\n                sell = first_number(obj.get("sellAmount"))\n                return buy - sell if buy is not None and sell is not None else None\n\n            output[date] = {\n                "date": date,\n                "foreignNetBuyValue": net_amount("foreigner"),\n                "institutionNetBuyValue": net_amount("institution"),\n                "individualNetBuyValue": net_amount("individual"),\n                "otherCorporationNetBuyValue": net_amount("otherCorporation"),\n                "marketFlowUpdatedAt": first_text(rec.get("updatedAt")),\n            }\n\n        if len(output) >= target_count:\n            break\n        cursor = extract_cursor(payload, ("nextUntil",))\n        if not cursor or cursor in seen:\n            break\n        seen.add(cursor)\n        until = cursor\n\n    if not output:\n        return pd.DataFrame()\n    return pd.DataFrame(output.values()).sort_values("date").tail(target_count).reset_index(drop=True)\n\ndef set_symbol_status(symbol, stage):\n    with _ACTIVE_STATUS_LOCK:\n        _ACTIVE_STATUS[symbol] = stage\n\ndef get_status_summary():\n    with _ACTIVE_STATUS_LOCK:\n        return Counter(_ACTIVE_STATUS.values())\n\ndef merge_on_date(base, frames):\n    out = base.copy()\n    for frame in frames:\n        if frame is None or frame.empty:\n            continue\n        f = frame.copy()\n        if "date" not in f.columns:\n            continue\n        # 같은 이름이 의도치 않게 겹치면 기존 base가 아닌 새 source를 명시적으로 설계해야 한다.\n        overlap = (set(out.columns) & set(f.columns)) - {"date"}\n        if overlap:\n            raise RuntimeError(f"merge column 충돌: {sorted(overlap)}")\n        out = out.merge(f, on="date", how="left", validate="one_to_one")\n    return out\n\ndef cached_toss_trend(symbol, stage, suffix, parser):\n    return cached_frame(\n        stage,\n        symbol,\n        lambda: fetch_trend_frame(symbol, suffix, parser, FLOW_COUNT),\n    )\n\ndef collect_short_lending_frames(symbol):\n    # This addition is for individual equities; ETF/index behavior is unchanged.\n    if symbol in ETF_SET:\n        return []\n    frames = []\n    for enabled, stage, suffix, parser in [\n        (COLLECT_TOSS_SHORT, \'toss_short\', \'short-selling\', parse_short_record),\n        (COLLECT_TOSS_LENDING, \'toss_lending\', \'securities-lending\', parse_lending_record),\n    ]:\n        if not enabled:\n            continue\n        set_symbol_status(symbol, stage)\n        frame = cached_toss_trend(symbol, stage, suffix, parser)\n        if frame.empty:\n            add_error(stage + \'_no_data\', symbol, suffix,\n                      \'조회 결과 없음 또는 수집 오류: 공표 여부와 오류 로그를 확인하세요. 0으로 대체하지 않습니다.\')\n        frames.append(frame)\n    return frames\n\ndef build_symbol_frame(symbol):\n    info = symbol_info.get(symbol, {})\n    is_etf = symbol in ETF_SET\n\n    set_symbol_status(symbol, "candles")\n    candles = cached_frame(\n        "toss_candles",\n        symbol,\n        lambda: get_candles_frame(symbol, TARGET_COUNT, False),\n    )\n    if candles.empty:\n        raise RuntimeError("Toss candle 데이터 없음")\n\n    start = candles["date"].min().replace("-", "")\n    end = candles["date"].max().replace("-", "")\n\n    frames = []\n\n    # Priority 3: Toss 투자자별 매매 + 보유량\n    if COLLECT_TOSS_INVESTOR:\n        set_symbol_status(symbol, "investor")\n        frames.append(cached_toss_trend(\n            symbol, "toss_investor", "investor-trading", parse_investor_record\n        ))\n\n    frames.extend(collect_short_lending_frames(symbol))\n\n    # Toss 네트워크 대기를 먼저 겹쳐 처리한 후 KRX 구간으로 진입합니다.\n    # KRX 자체는 krx_retry 내부 lock에 의해 전체 worker에서 1개씩만 호출됩니다.\n\n    if is_etf and COLLECT_KRX_ETF:\n        set_symbol_status(symbol, "krx_etf_reference")\n        frames.append(cached_frame(\n            "krx_etf_reference",\n            symbol,\n            lambda: get_krx_etf_reference_frame(symbol, start, end),\n        ))\n        set_symbol_status(symbol, "krx_etf_investor")\n        frames.append(cached_frame(\n            "krx_etf_investor_value",\n            symbol,\n            lambda: get_krx_etf_investor_value_frame(symbol, start, end),\n        ))\n\n    df = merge_on_date(candles, frames)\n\n    # 메타데이터\n    df.insert(0, "symbol", symbol)\n    df.insert(1, "name", first_text(info.get("name"), symbol))\n    df.insert(2, "market", first_text(info.get("market")))\n    df.insert(3, "securityType", "ETF" if is_etf else "STOCK")\n    df["priceSource"] = "TOSS_ADJUSTED_CANDLE"\n\n    # Priority 1 canonical exact fields.\n    if is_etf:\n        exact_tv = df["etfTradingValue"] if "etfTradingValue" in df.columns else pd.Series(pd.NA, index=df.index)\n        exact_cap = df["etfMarketCap"] if "etfMarketCap" in df.columns else pd.Series(pd.NA, index=df.index)\n        listed = df["etfListedUnits"] if "etfListedUnits" in df.columns else pd.Series(pd.NA, index=df.index)\n        exact_source = "KRX_ETF"\n    else:\n        exact_tv = df["krxTradingValue"] if "krxTradingValue" in df.columns else pd.Series(pd.NA, index=df.index)\n        exact_cap = df["krxMarketCap"] if "krxMarketCap" in df.columns else pd.Series(pd.NA, index=df.index)\n        listed = df["krxListedShares"] if "krxListedShares" in df.columns else pd.Series(pd.NA, index=df.index)\n        exact_source = "KRX_STOCK"\n\n    exact_tv_num = pd.to_numeric(exact_tv, errors="coerce")\n    close_num = pd.to_numeric(df["close"], errors="coerce")\n    volume_num = pd.to_numeric(df["volume"], errors="coerce")\n\n    df["tradingValue"] = exact_tv_num\n    proxy_mask = df["tradingValue"].isna()\n    df.loc[proxy_mask, "tradingValue"] = close_num[proxy_mask] * volume_num[proxy_mask]\n    df["tradingValueSource"] = exact_source\n    df.loc[proxy_mask, "tradingValueSource"] = "CLOSE_X_VOLUME_PROXY"\n\n    # 과거 시가총액은 KRX exact가 없으면 null. 현재 발행주식수×과거종가 금지.\n    df["marketCap"] = pd.to_numeric(exact_cap, errors="coerce")\n    df["listedShares"] = pd.to_numeric(listed, errors="coerce")\n    df["marketCapSource"] = pd.NA\n    df.loc[df["marketCap"].notna(), "marketCapSource"] = exact_source\n\n    # KRX exact 금액이 있으면 우선 사용합니다.\n    value_cols = [\n        "foreignNetBuyValue", "institutionNetBuyValue",\n        "individualNetBuyValue", "otherCorporationNetBuyValue",\n    ]\n    for c in value_cols:\n        if c not in df.columns:\n            df[c] = pd.NA\n\n    df["investorValueSource"] = pd.NA\n    exact_value_present = df[\n        ["foreignNetBuyValue", "institutionNetBuyValue"]\n    ].notna().all(axis=1)\n    df.loc[\n        exact_value_present,\n        "investorValueSource"\n    ] = "KRX_ETF" if is_etf else "KRX_STOCK"\n\n    partial_exact = df[value_cols].notna().any(axis=1) & ~exact_value_present\n    df.loc[partial_exact, "investorValueSource"] = "KRX_ETF_PARTIAL" if is_etf else "KRX_STOCK_PARTIAL"\n    # 운영 주식 스크리닝에서만 V1 proxy fallback을 유지합니다.\n    # ETF 및 exact 백테스트의 KRX 금액 결측은 그대로 둡니다.\n    if not is_etf:\n        for value_col, volume_col in [\n            ("foreignNetBuyValue", "foreignNetBuyVolume"),\n            ("institutionNetBuyValue", "institutionNetBuyVolume"),\n            ("individualNetBuyValue", "individualNetBuyVolume"),\n            ("otherCorporationNetBuyValue", "otherCorporationNetBuyVolume"),\n        ]:\n            if volume_col in df:\n                q = pd.to_numeric(df[volume_col], errors="coerce")\n                mask = df[value_col].isna() & q.notna() & close_num.notna()\n                df.loc[mask, value_col] = q[mask] * close_num[mask]\n        proxy_mask = df[value_cols].notna().any(axis=1) & df["investorValueSource"].isna()\n        df.loc[proxy_mask, "investorValueSource"] = "TOSS_NET_VOLUME_X_CLOSE_PROXY"\n\n    # Toss 종목 수급은 KRX+NXT 통합 거래량. 금액으로 환산하지 않는다.\n    investor_volume_cols = [\n        "individualNetBuyVolume", "foreignNetBuyVolume",\n        "institutionNetBuyVolume", "otherCorporationNetBuyVolume",\n    ]\n    present_cols = [c for c in investor_volume_cols if c in df.columns]\n    df["investorVolumeSource"] = pd.NA\n    if present_cols:\n        m = df[present_cols].notna().any(axis=1)\n        df.loc[m, "investorVolumeSource"] = "TOSS_KRX_NXT"\n\n    df["programTradeSource"] = pd.NA\n    if "programNetBuyVolume" in df.columns:\n        df.loc[df["programNetBuyVolume"].notna(), "programTradeSource"] = "TOSS_KRX_ONLY"\n\n    return df\n\ndef _completion_quality(frame, expected_symbol=None):\n    result = {\'ok\':False, \'reason\':None, \'metrics\':{}}\n    required = {\'symbol\',\'date\',\'open\',\'high\',\'low\',\'close\',\'volume\'}\n    if not isinstance(frame, pd.DataFrame) or frame.empty or not required.issubset(frame.columns):\n        result[\'reason\'] = \'empty_or_missing_price_columns\'\n        return result\n    symbols = set(frame[\'symbol\'].astype(str).str.upper())\n    if len(symbols) != 1 or (expected_symbol and symbols != {expected_symbol}):\n        result[\'reason\'] = \'unexpected_symbol\'\n        return result\n    if frame.duplicated([\'symbol\',\'date\']).any():\n        result[\'reason\'] = \'duplicate_symbol_date\'\n        return result\n    dates = pd.to_datetime(frame[\'date\'], errors=\'coerce\')\n    if dates.isna().any() or (COLLECTION_CUTOFF_DATE and (dates > pd.Timestamp(COLLECTION_CUTOFF_DATE)).any()):\n        result[\'reason\'] = \'invalid_or_future_date\'\n        return result\n    def num(col):\n        return pd.to_numeric(frame.get(col, pd.Series(float(\'nan\'), index=frame.index)), errors=\'coerce\')\n    def source(col, value):\n        return frame.get(col, pd.Series(\'\', index=frame.index)).eq(value).fillna(False)\n    def _recent_latest_krx_gap_allowed():\n        if not globals().get("ALLOW_RECENT_KRX_PUBLICATION_LAG", True):\n            return False\n        if globals().get("_OFFICIAL_FAILURE"):\n            return False\n        if dates.isna().all():\n            return False\n        latest_date = dates.max()\n        now_date = pd.Timestamp.now(tz="Asia/Seoul").tz_localize(None).normalize()\n        lag_days = int((now_date - latest_date.normalize()).days)\n        return 0 <= lag_days <= int(globals().get("KRX_PUBLICATION_LAG_MAX_CALENDAR_DAYS", 7))\n\n    latest_date_mask = dates.eq(dates.max()) if not dates.isna().all() else pd.Series(False, index=frame.index)\n    krx_eval_mask = pd.Series(True, index=frame.index)\n    latest_krx_pending = False\n    price_ok = (num(\'open\').gt(0) & num(\'high\').gt(0) & num(\'low\').gt(0) & num(\'close\').gt(0)\n                & num(\'volume\').ge(0) & num(\'high\').ge(num(\'low\'))\n                & num(\'high\').ge(num(\'open\')) & num(\'high\').ge(num(\'close\'))\n                & num(\'low\').le(num(\'open\')) & num(\'low\').le(num(\'close\')))\n    checks = {\'priceCoverage\':(price_ok, 0.95)}\n    is_etf = next(iter(symbols)) in ETF_SET\n    if is_etf and COLLECT_KRX_ETF:\n        # KRX가 아직 최신 거래일을 공표하지 않았다면 그 최신 1일만 exact coverage 분모에서 제외합니다.\n        # 인증/네트워크 장애(_OFFICIAL_FAILURE)는 절대 이 예외로 숨기지 않습니다.\n        latest_krx_complete = (\n            source(\'tradingValueSource\',\'KRX_ETF\')\n            & num(\'etfNav\').gt(0)\n            & num(\'etfMarketCap\').gt(0)\n            & num(\'etfListedUnits\').gt(0)\n        )\n        latest_rows_have_krx = bool(latest_krx_complete.loc[latest_date_mask].any()) if latest_date_mask.any() else True\n        if (not latest_rows_have_krx) and _recent_latest_krx_gap_allowed():\n            krx_eval_mask = ~latest_date_mask\n            latest_krx_pending = True\n            result.setdefault(\'warnings\', []).append(\'latest_krx_publication_pending\')\n        if krx_eval_mask.any():\n            checks.update({\n                \'exactTradingValue\':((source(\'tradingValueSource\',\'KRX_ETF\') & num(\'tradingValue\').ge(0)).loc[krx_eval_mask], EXACT_MIN_COVERAGE),\n                \'etfNav\':(num(\'etfNav\').gt(0).loc[krx_eval_mask], EXACT_MIN_COVERAGE),\n                # AUM은 KRX가 장 마감 직후/단일일 조회에서 비어 있을 수 있어\n                # 스크리닝에서는 AUM을 선택 품질항목으로 기록합니다.\n                \'etfMarketCap\':(num(\'etfMarketCap\').gt(0).loc[krx_eval_mask], EXACT_MIN_COVERAGE),\n                \'etfListedUnits\':(num(\'etfListedUnits\').gt(0).loc[krx_eval_mask], EXACT_MIN_COVERAGE),\n                \'exactInvestorValue\':((source(\'investorValueSource\',\'KRX_ETF\') & num(\'foreignNetBuyValue\').notna()\n                                      & num(\'institutionNetBuyValue\').notna()).loc[krx_eval_mask], EXACT_MIN_COVERAGE),\n            })\n        else:\n            result.setdefault(\'warnings\', []).append(\'krx_exact_pending_all_observed_rows\')\n        _aum_mask = num(\'etfNetAssetTotalAmount\').gt(0).fillna(False)\n        aum_cov = float(_aum_mask.loc[krx_eval_mask].mean()) if krx_eval_mask.any() else float("nan")\n        result[\'metrics\'][\'etfAum\'] = aum_cov\n        if pd.notna(aum_cov) and aum_cov < EXACT_MIN_COVERAGE:\n            result.setdefault(\'warnings\', []).append(\'etfAum_optional_missing_screening\')\n    if not is_etf:\n        for enabled, label, column in [\n            (COLLECT_TOSS_SHORT, \'tossShort\', \'shortSellingVolume\'),\n            (COLLECT_TOSS_LENDING, \'tossLending\', \'lendingBalanceQuantity\'),\n        ]:\n            if enabled:\n                cov = float(num(column).notna().mean())\n                result[\'metrics\'][label] = cov\n                if cov < EXACT_MIN_COVERAGE:\n                    result.setdefault(\'warnings\', []).append(label + \'_missing_or_publication_pending\')\n    # V2.1: Toss 수급수량은 보조지표. 장기 결측으로 KRX 핵심 데이터 전체를 버리지 않습니다.\n    toss_mask = num(\'foreignNetBuyVolume\').notna() & num(\'institutionNetBuyVolume\').notna()\n    result[\'metrics\'][\'tossInvestorVolume\'] = float(toss_mask.mean())\n    result.setdefault(\'warnings\', [])\n    if COLLECT_TOSS_INVESTOR and float(toss_mask.mean()) < EXACT_MIN_COVERAGE:\n        result[\'warnings\'].append(\'tossInvestorVolume_optional_missing\')\n    # 주식 스크리닝은 최근 20거래일 수급수량의 coverage를 확인합니다.\n    if not is_etf:\n        recent = frame.assign(_parsed_date=dates).sort_values(\'_parsed_date\').tail(20).index\n        checks[\'screeningInvestorVolume\'] = (toss_mask.loc[recent], EXACT_MIN_COVERAGE)\n    checks.pop(\'exactInvestorValue\', None)\n    result.setdefault(\'warnings\', []).append(\'individualInvestorAmount_not_provided_by_official_api\')\n\n    failed = []\n    for name, (mask, minimum) in checks.items():\n        cov = float(mask.fillna(False).mean())\n        result[\'metrics\'][name] = cov\n        if cov < minimum:\n            failed.append(f\'{name}_coverage_below_{minimum:.2f}:{cov:.3f}\')\n    result[\'ok\'] = not failed\n    result[\'reason\'] = \';\'.join(failed) or None\n    return result\n\n\nfrom collections import Counter\nBASE = "https://openapi.tossinvest.com"\nOPENAPI_SPEC_URL = BASE + "/openapi-docs/latest/openapi.json"\nTARGET_COUNT = FLOW_COUNT = 1\nTOSS_WORKERS = 1\nTOSS_CONNECT_TIMEOUT = 5\nTOSS_READ_TIMEOUT = 15\nTOSS_MAX_RETRIES = 5\nTOSS_CHART_GAP = .15\nTOSS_TREND_GAP = .12\nTOSS_STOCK_GAP = .35\nTOSS_INDICATOR_GAP = .12\nCOLLECTION_STOP_EVENT = threading.Event()\n_ACTIVE_STATUS = {}\n_ACTIVE_STATUS_LOCK = threading.Lock()\nCOLLECT_TOSS_INVESTOR = COLLECT_TOSS_SHORT = COLLECT_TOSS_LENDING = True\nCOLLECT_KRX_ETF = True\nEXACT_MIN_COVERAGE = .80\nCOLLECTION_CUTOFF_DATE = COLLECTION_CUTOFF_BEFORE = None\n\ndef cached_frame(stage, symbol, loader, refresh=None):\n    # Per-symbol in-process stage reuse only. Durable checkpoints are verified CSV in collector.py.\n    key = (stage,symbol,TARGET_COUNT,FLOW_COUNT,COLLECTION_CUTOFF_DATE)\n    if key not in _stage_frames: _stage_frames[key] = loader()\n    return _stage_frames[key].copy()\n_stage_frames = {}\n', 'collector.py': '"""Separate mounted-Drive collector. Immutable baseline + ordered delta layers; merging is TS engine only."""\nfrom __future__ import annotations\nimport csv, io, json, os, re, time, uuid, zipfile\nfrom contextlib import contextmanager\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom history import BASELINE_SHA, atomic, checked, require, sha\nVERSION=\'drive-history-collector-v7.0-preview\'\n\ndef load_observations(baseline, manifest, delta_paths=()):\n    # This is coverage planning, NOT a pandas replacement for the field-level official parser.\n    dates={}; metadata={}; last_price={}; latest_values={}\n    files=[Path(baseline)/s[\'member\'] for s in manifest[\'sources\']]+list(delta_paths)\n    for file in files:\n        with file.open(encoding=\'utf-8-sig\',newline=\'\') as f:\n            for r in csv.DictReader(f):\n                symbol=r[\'symbol\']; day=r[\'date\']; kind=r.get(\'type\',r.get(\'securityType\',\'\'))\n                dates.setdefault(symbol,set()).add(day)\n                if symbol not in metadata or day>=metadata[symbol][\'date\']:\n                    metadata[symbol]={\'name\':r.get(\'name\',symbol),\'market\':r.get(\'market\',\'\'),\'type\':kind,\'date\':day}\n                try: positive=float(r.get(\'close\') or 0)>0\n                except ValueError: positive=False\n                if positive: last_price[symbol]=max(day,last_price.get(symbol,\'\'))\n                # Retain only observed price fields for correction detection. Never fill analytical data here.\n                key=(symbol,day)\n                values=latest_values.setdefault(key,{})\n                for field in (\'open\',\'high\',\'low\',\'close\',\'volume\'):\n                    if r.get(field,\'\').strip(): values[field]=r[field]\n    return dates,metadata,last_price,latest_values\n\ndef incremental_plan(dates, universe, sessions, overlap=5, pending_dates=()):\n    require(sessions and sessions==sorted(set(sessions)),\'Invalid provider calendar\')\n    require(isinstance(overlap,int) and overlap>=1,\'Overlap must be positive\')\n    plans=[]\n    for symbol in universe:\n        known=dates.get(symbol,set()); require(known,\'New symbol requires an explicitly reviewed history bootstrap\')\n        first=min(known); expected=[d for d in sessions if d>=first]\n        gaps=[d for d in expected if d not in known]\n        recheck=expected[-overlap:]+[d for d in pending_dates if d>=first and d<=sessions[-1]]\n        wanted=sorted(set(gaps+recheck)); start=wanted[0] if wanted else sessions[-1]\n        plans.append({\'symbol\':symbol,\'firstObserved\':first,\'lastObserved\':max(known),\'missingSessions\':gaps,\n                      \'recheckSessions\':sorted(set(recheck)),\'start\':start,\'end\':sessions[-1],\n                      \'providerSessions\':sum(d>=start for d in sessions)})\n    return plans\n\n@contextmanager\ndef run_lock(directory):\n    """Same mounted filesystem writer exclusion; never promotes a shared latest/baseline pointer."""\n    directory=Path(directory); directory.mkdir(parents=True,exist_ok=True); lock=directory/\'writer.lock\'\n    try: fd=os.open(lock,os.O_CREAT|os.O_EXCL|os.O_WRONLY,0o600)\n    except FileExistsError: raise RuntimeError(\'This run is locked; resume after the other writer stops. Stale locks require inspection.\') from None\n    try:\n        os.write(fd,str(os.getpid()).encode());os.close(fd);yield\n    finally: lock.unlink(missing_ok=True)\n\ndef frame_csv(frame): return frame.to_csv(index=False,na_rep=\'\',lineterminator=\'\\n\').encode(\'utf-8-sig\')\ndef read_frame(file):\n    import pandas as pd\n    return pd.read_csv(file,dtype={\'symbol\':str,\'date\':str},keep_default_na=False)\n\ndef publication_evidence(p, sessions, checked_at):\n    latest=sessions[-1]; previous=sessions[-2]; markets={};pending=False\n    for market in (\'KOSPI\',\'KOSDAQ\',\'ETF\'):\n        frame=p.official_day(market,latest)\n        qa=next(q for q in p._OFFICIAL_QA if q[\'date\']==latest and q[\'market\']==market)\n        if len(frame): last=latest;status=\'EXACT_READY\'\n        else:\n            require(len(p.official_day(market,previous))>0,\'KRX publication gap is not a verified one-session delay\')\n            last=previous;status=\'PENDING_PUBLICATION\';pending=True\n        markets[market]={\'provider\':\'KRX_OPENAPI\',\'targetDate\':latest,\'lastPublishedDate\':last,\'status\':status,\n                        \'rawRows\':qa[\'rawRows\'],\'usableRows\':qa[\'usableRows\'],\'responseStatus\':qa[\'status\'],\n                        \'referenceOnlyPrices\':qa[\'status\']==\'reference_only_no_prices\'}\n    return {\'version\':\'krx-publication-readiness-v1\',\'phase\':\'PRELIMINARY\' if pending else \'FINAL\',\n            \'calendar\':{\'latestSession\':latest,\'previousSession\':previous,\'sessions\':sessions,\'checkedAt\':checked_at,\n                        \'source\':\'TOSS_BOTH_MARKET_INDEX_CANDLES\'},\'markets\':markets}\n\ndef collect(baseline, manifest, root, *, cutoff, run_id=None, excluded_symbols=(), overlap=5, parent=None):\n    """Uses the existing approved Toss/KRX endpoints in Colab; no Supabase credential is needed."""\n    import pandas as pd\n    import provider as p\n    run_id=run_id or uuid.uuid4().hex;require(re.fullmatch(r\'[A-Za-z0-9_-]{8,128}\',run_id),\'Invalid run ID\')\n    require(re.fullmatch(r\'\\d{4}-\\d{2}-\\d{2}\',cutoff),\'Invalid price cutoff\')\n    require(not parent,\'Parent delta promotion is disabled until distributed CAS is validated; baseline preview only\')\n    run=Path(root)/\'runs\'/run_id; run.mkdir(parents=True,exist_ok=True)\n    with run_lock(run):\n        code_hashes={n:sha((Path(__file__).parent/n).read_bytes()) for n in [\'history.py\',\'provider.py\',\'collector.py\',\'colab_support.py\']}\n        request_key=sha(json.dumps({\'codeHashes\':code_hashes,\'cutoff\':cutoff,\'excluded\':sorted(excluded_symbols),\'overlap\':overlap,\'baseline\':BASELINE_SHA,\'version\':VERSION},sort_keys=True).encode())\n        request_path=run/\'request-identity.txt\'\n        if request_path.exists():require(request_path.read_text()==request_key,\'Run ID already has different collection parameters\')\n        else:atomic(request_path,request_key.encode())\n        # A completed run is immutable. Reusing its ID returns exact verified artifacts, not a new collection.\n        final=run/\'complete-manifest.json\'\n        if final.exists():\n            done=json.loads(final.read_text());require(done[\'baselineSha256\']==BASELINE_SHA and done[\'cutoffDateInclusive\']==cutoff,\'Completed run has different parameters\')\n            for f in done[\'files\']:checked((run/f[\'name\']).read_bytes(),f)\n            return done\n        created_path=run/\'created-at.txt\'\n        if not created_path.exists():atomic(created_path,datetime.now(timezone.utc).isoformat().encode())\n        created=created_path.read_text(); p.CLIENT_ID=os.environ[\'TOSS_CLIENT_ID\'];p.CLIENT_SECRET=os.environ[\'TOSS_CLIENT_SECRET\']\n        p.COLLECTION_CUTOFF_DATE=cutoff;p.COLLECTION_CUTOFF_BEFORE=cutoff+\'T23:59:59+09:00\'\n        p.OUTPUT_ROOT=run;p.REFRESH_CACHE=True\n        os.environ[\'CLOUDTREND_STATE_ROOT\']=str(run)\n        p.client=p.TossClient();p.validate_toss_openapi(strict=True)\n        dates,metadata,last_price,old_prices=load_observations(baseline,manifest)\n        require(cutoff>max(max(v) for v in dates.values()),\'New preview runs must be later than the immutable baseline; same-session revisions require the publication adapter\')\n        # Recover the actual configured universe, never expand to old inactive/research instruments.\n        previous=json.loads(manifest[\'sources\'][-1][\'sourceRecord\'][\'validation_result\'][\'krPublication\'][\'manifestText\'])\n        exclusions=set(excluded_symbols)\n        configured=previous[\'requestedSymbols\'];universe=[s for s in configured if s not in exclusions]\n        require(universe and all(s in metadata for s in universe),\'Configured universe metadata is incomplete\')\n        p.ETF_SET={s for s in universe if metadata[s][\'type\']==\'ETF\'};p.symbol_info=metadata\n        calendar_count=max(520,(pd.Timestamp(cutoff)-pd.Timestamp(min(min(v) for v in dates.values()))).days+20)\n        require(calendar_count<=6000,\'Calendar exceeds reviewed collector bound\')\n        probes={i:p.get_candles_frame(i,calendar_count,True) for i in (\'KOSPI\',\'KOSDAQ\')}\n        calendars=[set(f.loc[pd.to_numeric(f.close,errors=\'coerce\').gt(0),\'date\']) for f in probes.values()]\n        require(calendars[0]==calendars[1] and len(calendars[0])>=2,\'Both market calendars must agree\')\n        sessions=sorted(calendars[0]);require(sessions[-1]==cutoff,\'Cutoff is not the latest actual completed session; use the date shown by the provider\')\n        pending_dates=[]\n        for source in manifest[\'sources\']:\n            contract=source[\'sourceRecord\'][\'validation_result\'].get(\'krPublication\',{})\n            if contract.get(\'phase\')==\'PRELIMINARY\':pending_dates.append(contract[\'asOfDate\'])\n        plans=incremental_plan(dates,universe,sessions,overlap,pending_dates)\n        atomic(run/\'collection-plan.json\',json.dumps({\'plans\':plans,\'lastActualPriceDates\':last_price,\'excludedSymbols\':sorted(exclusions)},ensure_ascii=False,indent=2).encode())\n        publication=publication_evidence(p,sessions,datetime.now(timezone.utc).isoformat())\n        expected=[];missing=[];price_excluded=[];corrections=[]\n        for n,plan in enumerate(plans):\n            symbol=plan[\'symbol\'];file=run/\'checkpoints\'/f\'{symbol}.csv\';receipt=file.with_suffix(\'.json\')\n            key=sha(json.dumps({\'plan\':plan,\'baseline\':BASELINE_SHA,\'phase\':publication[\'phase\'],\'version\':VERSION},sort_keys=True).encode())\n            if file.exists() and receipt.exists():\n                saved=json.loads(receipt.read_text());require(saved[\'key\']==key,\'Checkpoint parameters changed; use a new run ID\')\n                checked(file.read_bytes(),saved);frame=read_frame(file)\n            else:\n                try:\n                    p.TARGET_COUNT=p.FLOW_COUNT=plan[\'providerSessions\'];p._stage_frames.clear()\n                    frame=p.build_symbol_frame(symbol)\n                    # Stock exact facts are fetched from the shared official market/day service, without price substitution.\n                    exact=p.official_symbol_frame(symbol,plan[\'start\'].replace(\'-\',\'\'),cutoff.replace(\'-\',\'\'),symbol in p.ETF_SET)\n                    if symbol not in p.ETF_SET and not exact.empty:\n                        frame=frame.drop(columns=[c for c in exact.columns if c!=\'date\' and c in frame],errors=\'ignore\').merge(exact,on=\'date\',how=\'left\',validate=\'one_to_one\')\n                        for src,dst in [(\'krxMarketCap\',\'marketCap\'),(\'krxListedShares\',\'listedShares\'),(\'krxTradingValue\',\'tradingValue\')]:\n                            values=pd.to_numeric(frame[src],errors=\'coerce\');mask=values.notna();frame.loc[mask,dst]=values[mask]\n                        frame.loc[pd.to_numeric(frame[\'marketCap\'],errors=\'coerce\').gt(0),\'marketCapSource\']=\'KRX_STOCK\'\n                        frame.loc[pd.to_numeric(frame[\'krxTradingValue\'],errors=\'coerce\').notna(),\'tradingValueSource\']=\'KRX_STOCK\'\n                    frame=frame.loc[frame.date.between(plan[\'start\'],cutoff)].copy()\n                    require(not frame.duplicated([\'symbol\',\'date\']).any(),\'Duplicate symbol/date\')\n                    latest=frame.loc[frame.date.eq(cutoff)]\n                    if latest.empty or not pd.to_numeric(latest.close,errors=\'coerce\').gt(0).all():\n                        price_excluded.append({\'symbol\':symbol,\'reason\':\'NO_POSITIVE_PRICE_ON_TARGET_SESSION\',\'lastActualPriceDate\':last_price.get(symbol)});continue\n                    require(set(plan[\'missingSessions\'])<=set(frame.date),\'Requested missing sessions remain unfilled; distinguish suspension/listing using provider evidence\')\n                    quality=p._completion_quality(frame,symbol);require(quality[\'ok\'],\'Existing collector quality contract failed\')\n                    market=\'ETF\' if symbol in p.ETF_SET else metadata[symbol][\'market\']\n                    ready=publication[\'markets\'].get(market)\n                    require(ready,\'Unrecognized configured market\')\n                    if ready[\'status\']==\'EXACT_READY\':\n                        require(pd.to_numeric(frame.loc[frame.date.eq(cutoff),\'marketCap\'],errors=\'coerce\').gt(0).all(),\'Published KRX current market cap missing\')\n                    else:\n                        frame.loc[frame.date.eq(cutoff),\'marketCapSource\']=\'PENDING_PUBLICATION\'\n                    raw=frame_csv(frame);atomic(file,raw);atomic(receipt,json.dumps({\'key\':key,\'bytes\':len(raw),\'sha256\':sha(raw),\'quality\':quality},ensure_ascii=False).encode())\n                except Exception:\n                    missing.append({\'symbol\':symbol,\'reason\':\'COLLECTION_OR_COVERAGE_FAILED\'});continue\n            for row in frame.to_dict(\'records\'):\n                old=old_prices.get((symbol,row[\'date\']),{})\n                for field in (\'open\',\'high\',\'low\',\'close\'):\n                    if field in old and pd.notna(row.get(field)) and abs(float(old[field])-float(row[field]))>1e-8:\n                        corrections.append({\'symbol\':symbol,\'date\':row[\'date\'],\'field\':field,\'before\':old[field],\'after\':str(row[field]),\'provider\':\'TOSS_ADJUSTED_CANDLE\',\'checkpointSha256\':sha(file.read_bytes())})\n            expected.append(file)\n            if (n+1)%10==0:print(f\'검증 완료 {n+1}/{len(plans)}; 누락 {len(missing)}; 당일 가격 제외 {len(price_excluded)}\')\n        # Index collection is independently bounded to earliest required delta; full baseline is kept for all indicators.\n        first=min(p[\'start\'] for p in plans)\n        for symbol,frame in probes.items():\n            frame=frame.loc[frame.date.ge(first)].copy();frame[\'symbol\']=symbol;frame[\'name\']=symbol;frame[\'market\']=\'INDEX\';frame[\'securityType\']=\'INDEX\';frame[\'priceSource\']=\'TOSS_MARKET_INDICATOR\'\n            flow=p.get_market_investor_flow_frame(symbol,sum(d>=first for d in sessions));frame=p.merge_on_date(frame,[flow]);frame[\'tradingValue\']=0;frame[\'tradingValueSource\']=\'NOT_APPLICABLE_INDEX\';frame[\'investorValueSource\']=\'TOSS_MARKET_INDICATOR_KRW\'\n            file=run/\'checkpoints\'/f\'{symbol}.csv\';atomic(file,frame_csv(frame));expected.append(file)\n        atomic(run/\'corrections.json\',json.dumps(corrections,ensure_ascii=False,indent=2).encode())\n        # Price restatement is quarantined: full adjusted history/event evidence must be reviewed before completion.\n        complete=not missing and not corrections\n        archive=run/\'delta.zip\';buf=io.BytesIO();entries=[]\n        with zipfile.ZipFile(buf,\'w\',zipfile.ZIP_DEFLATED) as z:\n            for f in expected:\n                raw=f.read_bytes();z.writestr(f.name,raw);entries.append({\'name\':f.name,\'bytes\':len(raw),\'sha256\':sha(raw)})\n        atomic(archive,buf.getvalue())\n        targets=[s for s in universe if s not in {x[\'symbol\'] for x in price_excluded}]\n        result={\'version\':VERSION,\'history_backend\':\'drive\',\'result_backend\':\'drive\',\'publish_dashboard\':False,\n                \'baselineVersion\':\'screening_history_baseline_20261107_v1\',\'baselineSha256\':BASELINE_SHA,\n                \'orderedSourceHashes\':[s[\'sha256\'] for s in manifest[\'sources\']],\'parentManifestSha256\':BASELINE_SHA,\n                \'runId\':run_id,\'createdAt\':created,\'codeCommit\':os.environ.get(\'CLOUDTREND_CODE_COMMIT\',\'UNKNOWN\'),\n                \'codeHashes\':code_hashes,\'rulesVersion\':\'repository-official-engine\',\'mode\':\'SCREENING\',\'complete\':complete,\n                \'actualLatestPriceDate\':cutoff,\'cutoffDateInclusive\':cutoff,\'requestedSymbols\':targets,\n                \'excludedSymbols\':sorted(exclusions),\'missingPriceExcludedSymbols\':price_excluded,\'missingSymbols\':[m[\'symbol\'] for m in missing],\n                \'missingSymbolCount\':len(missing),\'missingIndexSymbols\':[], \'correctionCount\':len(corrections),\n                \'latestKrxPublicationPending\':publication[\'phase\']==\'PRELIMINARY\',\'latestKrxPendingDate\':cutoff if publication[\'phase\']==\'PRELIMINARY\' else None,\n                \'krxPublicationEvidence\':publication,\'deltaMembers\':entries,\n                \'files\':[{\'name\':\'delta.zip\',\'bytes\':archive.stat().st_size,\'sha256\':sha(archive.read_bytes())}],\n                \'promotionStatus\':\'DISABLED_PREVIEW_NO_LATEST_POINTER\'}\n        atomic(run/(\'complete-manifest.json\' if complete else \'incomplete-manifest.json\'),json.dumps(result,ensure_ascii=False,indent=2).encode())\n        print(f\'완료={complete}, 가격 제외={len(price_excluded)}, 누락={len(missing)}, 수정주가 변경 격리={len(corrections)}\')\n        return result\n', 'colab_support.py': '"""Colab credential adapter and immutable package publication. Never updates baseline/latest."""\nimport json,time\nfrom pathlib import Path\nfrom history import Drive,require,sha,atomic\nclass ColabDrive(Drive):\n    def __init__(self):\n        from google.colab import auth\n        import google.auth\n        auth.authenticate_user()\n        self.google_credentials,_=google.auth.default()\n        self.token,self.until,self.reads=\'\',0,[]\n    def refresh(self):\n        from google.auth.transport.requests import Request\n        self.google_credentials.refresh(Request())\n        self.token,self.until=self.google_credentials.token,time.monotonic()+1200\n\ndef children(drive,parent,name):\n    require("\'" not in name,\'Invalid name\')\n    query={\'q\':f"\'{parent}\' in parents and name=\'{name}\' and trashed=false",\'fields\':\'files(id,name,mimeType)\',\'pageSize\':100}\n    found=json.loads(drive.call(\'files\',query=query))[\'files\'];require(len(found)<=1,\'Duplicate immutable artifacts/folders; inspect before resuming\');return found\n\ndef immutable_upload(drive,parent,name,raw):\n    found=children(drive,parent,name)\n    if found:\n        require(sha(drive.download(found[0][\'id\']))==sha(raw),\'Same immutable artifact name contains different bytes\');return found[0]\n    return drive.upload(parent,name,raw)\n\ndef export_completed(run_path, *, parent_folder=\'14rxxO_vmmxeuSAQTZ1B7GwlTVFd8LMI-\', drive=None):\n    drive=drive or ColabDrive();run=Path(run_path);m=json.loads((run/\'complete-manifest.json\').read_text())\n    require(m[\'complete\'] is True,\'Run is incomplete\')\n    name=\'drive_history_v7_run_\'+m[\'runId\'];found=children(drive,parent_folder,name)\n    if found:folder=found[0]\n    else:folder=json.loads(drive.call(\'files\',data=json.dumps({\'name\':name,\'mimeType\':\'application/vnd.google-apps.folder\',\'parents\':[parent_folder]}).encode(),content_type=\'application/json\',query={\'fields\':\'id,name,webViewLink\'}))\n    # API upload + authenticated full readback precede a complete API manifest.\n    for file in m[\'files\']:\n        raw=(run/file[\'name\']).read_bytes();require(sha(raw)==file[\'sha256\'] and len(raw)==file[\'bytes\'],\'Run artifact changed\')\n        saved=immutable_upload(drive,folder[\'id\'],file[\'name\'],raw);file[\'fileId\']=saved[\'id\']\n    raw=json.dumps(m,ensure_ascii=False,indent=2).encode()\n    saved=immutable_upload(drive,folder[\'id\'],\'complete-manifest.json\',raw)\n    launch={\'run_manifest_id\':saved[\'id\'],\'run_manifest_sha256\':sha(raw),\'result_folder_id\':folder[\'id\'],\'publish_dashboard\':False}\n    atomic(run/\'actions-inputs.json\',json.dumps(launch,indent=2).encode())\n    print(json.dumps(launch,indent=2));return launch\n'}
EXPECTED = {'history.py': '3defd698cc35d7bf42074f11578a7895dcf865f54a66f2df801b01fd40dfa61a', 'provider.py': 'aa08b5be3d74eac2886f82a75855428bbda304972acb93247c86dc77235417c4', 'collector.py': 'a4ad60f1c41fbbe91748770f26fd37825f8b02ead6bb05c5844a75497dee00ca', 'colab_support.py': '823cd3b5e55285c77b6737afd31f90dbb6d7e6c89cd74fd80017a0450cdbdaff'}
CODE_ROOT = Path('/content/cloudtrend-drive-history-v7')
CODE_ROOT.mkdir(parents=True,exist_ok=True)
for name, text in FILES.items():
    assert hashlib.sha256(text.encode()).hexdigest() == EXPECTED[name]
    (CODE_ROOT/name).write_text(text)
sys.path.insert(0,str(CODE_ROOT))
os.environ['CLOUDTREND_CODE_COMMIT']='repository-v7-preview; embedded code hashes in code-manifest.json'
RUN_ROOT.mkdir(parents=True,exist_ok=True)
(RUN_ROOT/'code-manifest.json').write_text(json.dumps(EXPECTED,indent=2))


In [ ]:
#@title 4. 과거 원천 검증 — Supabase 다운로드 없음
from history import prepare_baseline
VERIFIED = Path('/content/cloudtrend-verified-drive-baseline-v7')
baseline_manifest = prepare_baseline(VERIFIED,mounted=BASELINE)
print('기준 원천 52개 검증 완료. 등록 순서를 유지했습니다.')
# Colab 재시작 후 캐시가 없어도 같은 검증을 다시 수행할 수 있습니다.


In [ ]:
#@title 5. 새 구간 수집 — Toss 허용 IP와 KRX 서비스 승인 필요
if DO_COLLECT:
    assert DATA_DATE, 'DATA_DATE에 실제 완료 거래일을 입력하세요.'
    for key in ['TOSS_CLIENT_ID','TOSS_CLIENT_SECRET','KRX_OPENAPI_KEY']:
        if not os.environ.get(key): os.environ[key] = userdata.get(key)
    import requests
    print('Toss 허용 IP 확인:',requests.get('https://api.ipify.org',timeout=15).text.strip())
    from collector import collect
    collection = collect(VERIFIED,baseline_manifest,RUN_ROOT,cutoff=DATA_DATE,
                         run_id=RUN_ID,excluded_symbols=EXCLUDED_SYMBOLS,overlap=OVERLAP_SESSIONS)
    print('완료 여부:',collection['complete'])
else:
    print('읽기 검증 완료. 실제 수집은 DO_COLLECT를 켜고 실행하세요.')


In [ ]:
#@title 6. 완료 package API 업로드·재다운로드 검증 후 Actions 입력 생성
if DO_COLLECT and collection['complete']:
    from colab_support import export_completed
    launch = export_completed(RUN_ROOT/'runs'/RUN_ID)
    print('GitHub → Actions → CloudTrend Drive history v7 → Run workflow')
    print('위 네 가지 값을 입력하세요. publish_dashboard=False를 유지하세요.')
    print('새 workflow가 main에 병합된 뒤 사용할 수 있습니다.')
elif DO_COLLECT:
    print('누락/정정 목록을 검토한 뒤 같은 runId로 재개하세요. 미완료 run은 Actions 입력으로 게시하지 않습니다.')


### 실행 범위와 남은 검증

이 노트북의 기본 검증과 로컬 비교는 Supabase 쓰기를 하지 않습니다. 새 Actions는 GET 경로를 기록하고 `source/screening`·`source/us-screening` 접근 및 Supabase 변경 요청을 차단합니다. 설정·이전 snapshot·승인된 ETF 지수 자료는 별도 읽기로 기록합니다.

현재 버전은 고정 baseline의 별도 검증용 run입니다. 부모 delta를 이어받은 장기 운영·동시 Colab의 전역 잠금·새 baseline 승격은 열지 않았습니다. 같은 run 폴더의 잠금 파일은 실행 중 중복을 막으며, 비정상 종료 시 작성자가 멈춘 것을 확인한 뒤 수동 정리가 필요합니다.

Supabase 프리플랜 안전성은 이 구현으로 판정하지 않습니다.
